In [2]:

from google.colab import output
output.enable_custom_widget_manager() # Necesario para Colab

import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import time

# --- 1. CONSTANTES FÍSICAS ---
radio_Tierra_m = 6371.0       # Radio terrestre (km)
H_observador_m = 1.70         # Altura de los ojos del observador (m)
c_zero_point = 14.18          # Calibración Lux (Magnitud)

# --- 2. MOTOR DE CÁLCULO FÍSICO ---
def calculo_fisico(I0_log, visibilidad, color_nm, ambiente, turbulencia, pitch):
    """
    Realiza los cálculos de magnitud aparente basándose en la física atmosférica.
    """

    # Simulación de carga (feedback visual)
    time.sleep(0.3)

    # CONFIGURACIÓN DEL ESPACIO LOG
    I0 = 10**I0_log # Convertir log a intensidad lineal
    distancias_m = np.linspace(160000, 50, 1500) # Vector de distancias (160km a 50m)
    d_km = distancias_m / 1000.0

    # HORIZONTE GEOMÉTRICO
    slider_theta = np.radians(3.0) # Ángulo de descenso (Glide slope)
    h_avion_m = distancias_m * np.sin(slider_theta)

    # Cálculo del límite visual por curvatura de la Tierra
    horizonte_km = (np.sqrt(2 * radio_Tierra_m * (H_observador_m / 1000.0)) +
                    np.sqrt(2 * radio_Tierra_m * (h_avion_m / 1000.0)))

    es_visible_geo = d_km < horizonte_km # Máscara booleana

    # DINÁMICA DE LA FUENTE (PITCH EFFECT)
    I_eff = I0

    if pitch: # Si el checkbox de pitch está activado
        desv_max = np.radians(4.0) # Desviación máxima a corta distancia
        # La desviación disminuye con la distancia (efecto geométrico)
        desviacion = desv_max * (1 - (distancias_m / 60000.0))
        desviacion = np.maximum(desviacion, 0) # Evitar valores negativos

        sigma_haz = np.radians(5.0) # Ancho del haz gaussiano
        # Fórmula de caída de intensidad gaussiana
        I_eff = I0 * np.exp(- (desviacion**2) / (2 * sigma_haz**2))

    # EXTINCIÓN ATMOSFÉRICA
    # Dispersión de Rayleigh (Depende de la longitud de onda - color)
    H_Rayleigh = 8000.0
    k_rayleigh = 0.012 * ((550.0 / color_nm)**4) / 1000.0

    # Dispersión de Mie (Depende de la visibilidad - aerosoles/niebla)
    H_Mie = 2000.0
    k_mie = (3.912 / visibilidad) / 1000.0

    # Cálculo del Espesor Óptico (Tau) - Integral de camino
    tau = ((k_rayleigh * H_Rayleigh + k_mie * H_Mie) / np.sin(slider_theta)) * \
          (1 - np.exp(-distancias_m * np.sin(slider_theta) / 3000.0))

    # FOTOMETRÍA (CÁLCULO FINAL)
    # Ley del cuadrado inverso + Atenuación atmosférica (Beer-Lambert)
    E_lux = (I_eff / (distancias_m**2 + 100.0)) * np.exp(-tau)
    magnitud = -2.5 * np.log10(np.maximum(E_lux, 1e-25)) - c_zero_point

    # TURBULENCIA Y CORTES
    if turbulencia:
        np.random.seed(int(time.time())) # Semilla aleatoria basada en el tiempo
        # El ruido (scintillation) aumenta con la distancia
        ruido = np.random.normal(0, 0.15, size=len(magnitud)) * (d_km / 10.0)**1.2
        magnitud += ruido

    # Aplicar corte de horizonte (lo que está detrás de la Tierra no se ve)
    magnitud[~es_visible_geo] = np.nan

    return d_km, magnitud

# --- 3. INTERFAZ GRÁFICA ---

# Configuración inicial de la figura (Plotly FigureWidget)
fig_widget = go.FigureWidget()
fig_widget.update_layout(
    title="Esperando datos...",
    template="plotly_dark",
    height=550,
    xaxis_title="Distancia (km)",
    yaxis_title="Magnitud Aparente (Escala Invertida)",
    yaxis=dict(autorange="reversed", range=[20, -16]), # Eje astronómico invertido
    xaxis=dict(range=[0, 150]),
    hovermode="x unified"
)
# Traza inicial vacía
fig_widget.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Avión', line=dict(width=3)))
# Línea de referencia del ojo humano
fig_widget.add_hline(y=6.0, line_dash="dot", line_color="white", annotation_text="Límite")

# CONTROLES (WIDGETS)
estilo = {'description_width': 'initial'}
layout_w = widgets.Layout(width='95%')

w_I0 = widgets.FloatSlider(value=6.0, min=4.0, max=7.5, step=0.1, description='Log Potencia', style=estilo, layout=layout_w)
w_vis = widgets.FloatSlider(value=40.0, min=1.0, max=100.0, step=1.0, description='Visibilidad (km)', style=estilo, layout=layout_w)
w_col = widgets.IntSlider(value=550, min=400, max=700, step=20, description='Color (nm)', style=estilo, layout=layout_w)
w_amb = widgets.Dropdown(options=['Noche', 'Crepúsculo', 'Día'], value='Noche', description='Ambiente', style=estilo, layout=layout_w)
w_turb = widgets.Checkbox(value=True, description='Turbulencia')
w_pitch = widgets.Checkbox(value=True, description='Pitch (Dinámica)')

barra_carga = widgets.IntProgress(value=0, max=100, bar_style='info', layout=widgets.Layout(width='98%'))
btn_calcular = widgets.Button(description=' ACTUALIZAR', button_style='primary', icon='plane', layout=widgets.Layout(width='98%'))

# LÓGICA DE ACTUALIZACIÓN
def actualizar_grafica(b):
    # Bloquear botón y reiniciar barra
    btn_calcular.disabled = True
    barra_carga.value = 0

    # Efecto visual de carga
    for i in range(0, 80, 20):
        barra_carga.value = i
        time.sleep(0.05)

    try:
        # Llamada a la función física con los valores de los sliders
        x, y = calculo_fisico(w_I0.value, w_vis.value, w_col.value,
                               w_amb.value, w_turb.value, w_pitch.value)

        # Actualización eficiente de la gráfica (Batch)
        with fig_widget.batch_update():
            fig_widget.data[0].x = x
            fig_widget.data[0].y = y

            # Cambiar color de línea según longitud de onda
            if w_col.value < 490: c = '#0066FF' # Azul
            elif w_col.value < 580: c = '#00FF00' # Verde
            elif w_col.value < 620: c = '#FFAA00' # Ámbar
            else: c = '#FF0000' # Rojo
            fig_widget.data[0].line.color = c

            # Configurar fondo según ambiente
            if w_amb.value == 'Noche':
                lim, bg = 6.0, '#0a0a0a'
            elif w_amb.value == 'Crepúsculo':
                lim, bg = 2.5, '#2d2d4a'
            else:
                lim, bg = -4.5, '#87CEEB' # Día

            fig_widget.layout.plot_bgcolor = bg
            fig_widget.layout.title = f"Entorno: {w_amb.value} | Visibilidad: {w_vis.value} km"

            # Rectángulo para simular zona invisible
            fig_widget.layout.shapes = [
                dict(type="rect", xref="paper", yref="y", x0=0, x1=1, y0=lim, y1=35,
                     fillcolor="black", opacity=0.5 if w_amb.value != 'Día' else 0.2, line_width=0, layer="below")
            ]

        barra_carga.value = 100 # Carga completa

    except Exception as e:
        print(f"Error: {e}")
    finally:
        btn_calcular.disabled = False # Reactivar botón

btn_calcular.on_click(actualizar_grafica)

# --- 4. DISPLAY FINAL ---
ui_final = widgets.VBox([
    widgets.HTML("<h3>✈️ Simulador Astrofísico (Variables Originales)</h3>"),
    widgets.GridBox([w_I0, w_vis, w_col, w_amb, w_turb, w_pitch], layout=widgets.Layout(grid_template_columns="repeat(2, 50%)")),
    btn_calcular, barra_carga, fig_widget
])

display(ui_final)
actualizar_grafica(None) # Ejecución inicial

**Constantes Físicas y Marco de Referencia **

Se establece:

1. Radio Terrestre: Se usa una aproximación esférica y se define como 6371.0 km.

2.  Punto Cero: En astrofísica, la magnitud aparente (m) se define en relación al flujo de referencia ($F_0$). El valor $C_{zp}$ ayuda a calibrar la ecuación de magnitud aparente: $m=-2.5 log_{10}(F)-C_{zp}$ para ajustar las unidades de la simulación ($W/m^2$) a escala de magnitudes visuales (la cual es logaritmica y por ende no aditiva).

```
radio_Tierra_m= 6371.0 #Radio terrestre [m]
H_observador_m= 1.70 #Altura de los ojos del observador[m]
c_zero_point = 14.18 #Calibración Lux (magnitud)
```

**Sistema**

Se crea un vector de distancias [m] que representa la posición del avión con respecto al obseervador. La intensidad lumínica de la fuente ($I_0$) es logarítmica, pero debe pasarse a una escala lineal [Watts/sr] para aplicar la ley inversa del cuadrado al flujo con respecto a la distancia.

$$I_{0}=10^{log_{portencia}}$$



```
def calcular_fisica(I0_log, visibilidad, color_nm, ambiente, turbulencia, pitch):
    time.sleep(0.3) # Simular carga

    # A. Configuración Inicial
    I0 = 10**I0_log
    distancias_m = np.linspace(160000, 50, 1500)
    d_km = distancias_m / 1000.0
```



* *distancias (d):* Se usa un arreglo con 1000 puntos desde 50km hasta 200m.

* *slide_theta:* Ángulo de descenso del avión. Para los aviones comerciales el estándar es 3°.

* *np.radianes:* Se requiere para hacer operaciones con radianes.

**Horizonte terrestre y oclusión**


Asumiendo que la Tierra es perfectamente esférica, la línea recta tangente a la esfera. El punto donde esa línea toca la superficie es el horizonte geométrico.

$d \approx \sqrt{2 R h_1} + \sqrt{2 R h_2}$
Se calcula la altura del avión ($h_{avion}$) asumiendo que la Tierra es perfectamente esférica y tomando una trayectoria de planeo constante ($\theta_{planeo}$). El límite de visibilidad geométrica está dado por la curvatura terrestre. Como se muestra en la Fig.119. La distancia al horizonte ($d_{hor}$) para un objeto a la altura $h_{1}$ observado desde una altura ($h_{2}$) se arpoxima por la suma de las distancias tangentes. Así, si la distancia del avión $d>d_{hor}$, el avión queda oculto detrás de la curvatura de la Tierra (oclusión).


Se utilizaron las siguientes ecuaciones para calcular la altura de la avión y la distancia del horizonte:


$$h_{\text{avion}} = d \cdot \sin(\theta_{\text{planeo}})$$

$$d_{\text{horizonte}} \approx \sqrt{2 R_{\oplus} h_{\text{obs}}} + \sqrt{2 R_{\oplus} h_{\text{avión}}}$$

```
# R = 6371 km. Se convierte altura de m a km
horizonte_km = (np.sqrt(2 * radio_Tierra_m * (H_observador_m / 1000.0)) +
                np.sqrt(2 * radio_Tierra_m * (h_avion_m / 1000.0)))

# Filtro booleano de visibilidad
es_visible_geo = d_km < horizonte_km
```

**Atenuación atmosférica (Extinción)**

Se tiene en cuenta lal dispersión de los fotones por las partículas atmosféricas. Para las moléculas de gas se usa la *dispersión Reyleigh* y para las partículas de aerosol se usa la *dispersión Mie*.


**Ley de Beer-Lambert**

$$E = \frac{I}{d^2} e^{-\tau}$$

```
# Dispersión de Rayleigh según el color (nm)
k_rayleigh = 0.012 * ((550.0 / color_nm)**4) / 1000.0

# Espesor óptico total considerando la trayectoria inclinada (slant path)
tau = ((k_rayleigh * H_Rayleigh + k_mie * H_Mie) / np.sin(theta_glide)) * \
      (1 - np.exp(-distancias_m * np.sin(theta_glide) / 3000.0))
```
**Fotometría y Magnitud**
Se necesita convertir la irradiancia (E) a escala logaritmica de magnitud aparente.


$$m = -2.5 \log_{10}(E) - C$$


```
magnitud = -2.5 * np.log10(np.maximum(E_lux, 1e-25)) - c_zero_point
```

**Efecto de Pitch (Cabeceo)**
Las luces del avión, en la realidad, no tienen la misma distribución en todos los sentidos (no son isótropas). Por ejemplo, si el avión inclina su morro (pitch), el eje del haz de luz se desvía del observador. Esta pérdida lumínica se puede modelar a partir de una distribución Gaussiana del haz.

$$I_{eff} = I_0 \cdot \exp\left( -\frac{\Delta\theta^2}{2\sigma^2} \right)$$



```
# PITCH EFFECT
    I_eff = I0
    if pitch:
        max_dev = np.radians(4.0)
        desviacion = max_dev * (1 - (distancias_m / 60000.0))
        desviacion = np.maximum(desviacion, 0)
        sigma_beam = np.radians(5.0)
        I_eff = I0 * np.exp(- (desviacion**2) / (2 * sigma_beam**2)    
```



**EXTINCIÓN ATMOSFÉRICA**
1. Dispersión Rayleigh: afecta longitudes de onda cortas ($λ^{-4}$)

$$k_{\lambda} \propto \lambda^{-4}$$

2. Dispersión de Mie (aerosoles): Depende de la visibilidad meteorológica (coeficiente de extinción). La profundidad óptica se integra a lo largo de la trayectoria.
$$k_{Mie} \approx \frac{3.912}{Visibilidad}$$

$$\tau(d) = \int_{0}^{d} (k_R e^{-z/H_R} + k_M e^{-z/H_M}) \, ds$$

```
# D. Atmósfera
    H_Rayleigh = 8000.0
    k_rayleigh = 0.012 * ((550.0 / color_nm)**4) / 1000.0
    H_Mie = 2000.0
    k_mie = (3.912 / visibilidad) / 1000.0
    
    tau = ((k_rayleigh * H_Rayleigh + k_mie * H_Mie) / np.sin(theta_glide)) * \
          (1 - np.exp(-distancias_m * np.sin(theta_glide) / 3000.0))
```

Fotometría Resultante y Escala de Magnitudes

$$E = \frac{I_{eff}}{d^2 + \epsilon} \cdot e^{-\tau}$$

$$m = -2.5 \log_{10}(E) - C_{zp}$$

Turbulencia Atmosférica y Corte Geométrico

$$m_{final} = m + \delta_{ruido}$$

$$\delta_{ruido} \sim \mathcal{N}(\mu=0, \sigma(d))$$

$$\sigma(d) = 0.15 \cdot \left( \frac{d}{10 \text{ km}} \right)^{1.2}$$

